In [8]:
import json
import os
import re
from typing import Callable, Dict

from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI


# ============================================================
# Memory and internal notes
# ============================================================

notes = [
    "Client needs delivery completed as soon as possible.",
    "Payment issue is currently blocking the delivery.",
    "Client requested a meeting tomorrow.",
    "Internal team can meet tomorrow after 11 AM.",
]

memory = []


# ============================================================
# Tools
# ============================================================

def search_notes(_: str) -> str:
    """
    Search internal notes for relevant information.
    """
    return " | ".join(notes)


def calculate_priority(user_goal: str) -> str:
    """
    Calculate whether the request is high or medium priority.
    """

    goal_lower = user_goal.lower()

    urgent_words = [
        "urgent",
        "payment",
        "blocked",
        "asap",
        "tomorrow",
    ]

    priority = (
        "high"
        if any(
            word in goal_lower
            for word in urgent_words
        )
        else "medium"
    )

    return f"Priority level: {priority}"


def draft_reply(user_goal: str) -> str:
    """
    Draft a reply to the client.
    """

    return (
        "Draft reply: Thank you for the update. "
        "I understand the issue and I can meet tomorrow "
        "after 11 AM to resolve the payment blocker."
    )


# ============================================================
# Tool registry
# ============================================================

TOOLS: Dict[str, Callable[[str], str]] = {
    "search_notes": search_notes,
    "calculate_priority": calculate_priority,
    "draft_reply": draft_reply,
}


# ============================================================
# Response helper
# ============================================================

def text_from_response(response) -> str:
    """
    Convert the LangChain response into plain text.

    Different versions of langchain-google-genai can return
    response.content either as a string or as a list of
    content blocks.
    """

    # --------------------------------------------------------
    # Get content
    # --------------------------------------------------------

    content = getattr(response, "content", response)

    # --------------------------------------------------------
    # Case 1: content is already a string
    # --------------------------------------------------------

    if isinstance(content, str):
        return content

    # --------------------------------------------------------
    # Case 2: content is a list of blocks
    # --------------------------------------------------------

    if isinstance(content, list):

        text_parts = []

        for block in content:

            # Example:
            #
            # {
            #     "type": "text",
            #     "text": '{"action": "..."}'
            # }

            if isinstance(block, dict):

                if block.get("type") == "text":

                    text = block.get("text")

                    if text:
                        text_parts.append(str(text))

            # Some versions may return objects instead
            # of dictionaries.

            elif hasattr(block, "text"):

                text = getattr(block, "text")

                if text:
                    text_parts.append(str(text))

        if text_parts:
            return "\n".join(text_parts)

    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------

    return str(content)


# ============================================================
# JSON extraction helper
# ============================================================

def extract_json(text: str) -> Dict[str, str]:
    """
    Extract a JSON object from model output.
    """

    text = text.strip()

    # --------------------------------------------------------
    # Remove Markdown code fences
    # --------------------------------------------------------

    text = re.sub(
        r"^```json\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"^```\s*",
        "",
        text,
    )

    text = re.sub(
        r"\s*```$",
        "",
        text,
    )

    text = text.strip()

    # --------------------------------------------------------
    # First attempt:
    # Parse the entire response
    # --------------------------------------------------------

    try:

        parsed = json.loads(text)

        if isinstance(parsed, dict):
            return parsed

    except json.JSONDecodeError:
        pass

    # --------------------------------------------------------
    # Second attempt:
    # Find JSON object inside the response
    # --------------------------------------------------------

    match = re.search(
        r"\{.*\}",
        text,
        re.DOTALL,
    )

    if match:

        json_text = match.group(0)

        try:

            parsed = json.loads(json_text)

            if isinstance(parsed, dict):
                return parsed

        except json.JSONDecodeError:
            pass

    # --------------------------------------------------------
    # Nothing could be parsed
    # --------------------------------------------------------

    return {}


# ============================================================
# Reasoning / action selection
# ============================================================

def choose_action(
    llm: ChatGoogleGenerativeAI,
    user_goal: str,
    current_state: str,
) -> Dict[str, str]:

    # --------------------------------------------------------
    # Create prompt
    # --------------------------------------------------------

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
You are a careful reasoning assistant.

Your task is to choose the single best next action.

Available actions:
{tool_list}

Available tools:

- search_notes: Search internal notes for relevant information.
- calculate_priority: Determine whether the request is high or medium priority.
- draft_reply: Draft a response to the client.

Return ONLY one valid JSON object.

The JSON MUST have exactly these fields:

{{
    "action": "one of the available tool names",
    "reason": "a short explanation of why this action is the best next step"
}}

Do not use Markdown.
Do not use ```json.
Do not add any text before or after the JSON.
""",
            ),
            (
                "human",
                "User goal: {goal}\n"
                "Earlier steps: {history}\n"
                "Current state: {state}",
            ),
        ]
    )

    # --------------------------------------------------------
    # Connect prompt and model
    # --------------------------------------------------------

    chain = prompt | llm

    # --------------------------------------------------------
    # Invoke model
    # --------------------------------------------------------

    response = chain.invoke(
        {
            "tool_list": ", ".join(TOOLS.keys()),
            "goal": user_goal,
            "history": (
                " | ".join(memory[-5:])
                if memory
                else "No earlier steps yet."
            ),
            "state": current_state,
        }
    )

    # --------------------------------------------------------
    # Convert response to text
    # --------------------------------------------------------

    response_text = text_from_response(response)

    # --------------------------------------------------------
    # Display raw text
    # --------------------------------------------------------

    print("\nRaw model text:")
    print(response_text)

    # --------------------------------------------------------
    # Extract JSON
    # --------------------------------------------------------

    parsed = extract_json(response_text)

    # --------------------------------------------------------
    # Display parsed JSON
    # --------------------------------------------------------

    print("\nParsed JSON:")
    print(parsed)

    # --------------------------------------------------------
    # Get action
    # --------------------------------------------------------

    action = parsed.get(
        "action",
        "draft_reply",
    )

    # --------------------------------------------------------
    # Validate action
    # --------------------------------------------------------

    if action not in TOOLS:
        action = "draft_reply"

    # --------------------------------------------------------
    # Get reason
    # --------------------------------------------------------

    reason = parsed.get(
        "reason",
        "No reason returned.",
    )

    # --------------------------------------------------------
    # Return decision
    # --------------------------------------------------------

    return {
        "action": action,
        "reason": reason,
    }


# ============================================================
# Main
# ============================================================

def main() -> None:

    # --------------------------------------------------------
    # Load environment variables
    # --------------------------------------------------------

    load_dotenv()

    # --------------------------------------------------------
    # Check API key
    # --------------------------------------------------------

    if not os.getenv("GOOGLE_API_KEY"):
        raise RuntimeError(
            "GOOGLE_API_KEY is missing in your .env file."
        )

    # --------------------------------------------------------
    # Create Gemini model
    # --------------------------------------------------------

    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash",
        temperature=0,
    )

    # --------------------------------------------------------
    # User goal
    # --------------------------------------------------------

    user_goal = (
        "A client wants a meeting tomorrow because "
        "a payment issue is blocking delivery. "
        "Help me reply properly."
    )

    # --------------------------------------------------------
    # Current state
    # --------------------------------------------------------

    current_state = (
        "We have a short list of internal notes "
        "and we need the next best step."
    )

    # --------------------------------------------------------
    # Display user goal
    # --------------------------------------------------------

    print("User goal:")
    print(user_goal)

    print(
        "\nChoosing the next action...\n"
    )

    # --------------------------------------------------------
    # Ask LLM to reason and choose action
    # --------------------------------------------------------

    decision = choose_action(
        llm,
        user_goal,
        current_state,
    )

    # --------------------------------------------------------
    # Display decision
    # --------------------------------------------------------

    print(
        f"\nChosen action: {decision['action']}"
    )

    print(
        f"Reason: {decision['reason']}\n"
    )

    # --------------------------------------------------------
    # Execute selected tool
    # --------------------------------------------------------

    tool_result = TOOLS[
        decision["action"]
    ](user_goal)

    # --------------------------------------------------------
    # Store action in memory
    # --------------------------------------------------------

    memory.append(
        f"Action used: {decision['action']}"
    )

    memory.append(
        f"Reason: {decision['reason']}"
    )

    memory.append(
        f"Tool result: {tool_result}"
    )

    # --------------------------------------------------------
    # Display tool output
    # --------------------------------------------------------

    print("Tool output:")
    print(tool_result)

    # --------------------------------------------------------
    # Display memory
    # --------------------------------------------------------

    print("\nCurrent memory:")

    for item in memory:
        print(f"- {item}")


# ============================================================
# Entry point
# ============================================================

if __name__ == "__main__":
    main()

User goal:
A client wants a meeting tomorrow because a payment issue is blocking delivery. Help me reply properly.

Choosing the next action...



c:\Users\sanjo\AppData\Local\Programs\Python\Python310\lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Raw model text:
{"action": "search_notes", "reason": "Search internal notes to gather context about the client's payment issue and meeting policies before proceeding."}

Parsed JSON:
{'action': 'search_notes', 'reason': "Search internal notes to gather context about the client's payment issue and meeting policies before proceeding."}

Chosen action: search_notes
Reason: Search internal notes to gather context about the client's payment issue and meeting policies before proceeding.

Tool output:
Client needs delivery completed as soon as possible. | Payment issue is currently blocking the delivery. | Client requested a meeting tomorrow. | Internal team can meet tomorrow after 11 AM.

Current memory:
- Action used: search_notes
- Reason: Search internal notes to gather context about the client's payment issue and meeting policies before proceeding.
- Tool result: Client needs delivery completed as soon as possible. | Payment issue is currently blocking the delivery. | Client requested a

One important point

This is actually a useful demonstration of what makes this an agent-style program rather than simply an LLM call.

Your program has:

User Goal

    ↓
LLM reasoning

    ↓
Choose an action

    ↓
Tool execution

    ↓
Store result in memory

In your current example:

User Goal

    ↓
Gemini

    ↓
"search_notes"

    ↓
search_notes()

    ↓
Internal notes

    ↓
Memory

The next stage of the agent can use that updated memory to make another decision, such as:

search_notes

      ↓
calculate_priority

      ↓
draft_reply

That is the important Agentic AI concept being demonstrated by this example: the LLM decides what action to take, while Python actually executes the action.